In [6]:
CUB_test_dir = "/home/mario/codes/datasynth-xai/data/CUB/test"
# CUB_test_dir 
# | 001.class_1/class_1_id1.jpg
# ... | 200.class_200/class_200_id1.jpg
#  Get 1 image for each class and name it as class_1.jpg

HAM1000_dir = "/home/mario/codes/datasynth-xai/data/HAM10000_processed/test"
# HAM1000_dir 
# | class_1/ ...
# ... | class_7/ ...
# get 20 images for each class and name them as class_1_1.jpg, class_1_2.jpg, ..., class_7_20.jpg

miniImagenet_classes = ["n01614925", "n01632777", "n01641577", "n01664065", "n01687978", "n01695060", "n01729322", "n01773157", "n01833805", "n01871265", "n01877812", "n01978455", "n01986214", "n02013706", "n02066245", "n02071294", "n02088466", "n02090379", "n02091635", "n02096437", "n02097130", "n02099429", "n02108089", "n02108915", "n02109047", "n02109525", "n02111889", "n02115641", "n02123045", "n02129165", "n02167151", "n02206856", "n02264363", "n02279972", "n02342885", "n02346627", "n02364673", "n02454379", "n02481823", "n02486261", "n02494079", "n02655020", "n02793495", "n02804414", "n02808304", "n02837789", "n02895154", "n02909870", "n02917067", "n02966687", "n03000684", "n03014705", "n03041632", "n03045698", "n03065424", "n03180011", "n03216828", "n03355925", "n03384352", "n03424325", "n03452741", "n03482405", "n03494278", "n03594734", "n03599486", "n03630383", "n03649909", "n03676483", "n03690938", "n03742115", "n03868242", "n03877472", "n03976467", "n03976657", "n03998194", "n04026417", "n04069434", "n04111531", "n04118538", "n04200800", "n04201297", "n04204347", "n04239074", "n04277352", "n04370456", "n04409515", "n04456115", "n04479046", "n04487394", "n04525038", "n04591713", "n04599235", "n07565083", "n07613480", "n07695742", "n07714571", "n07717410", "n07753275", "n10148035", "n12768682"]

ImageNet_dir = "/home/mario/codes/datasynth-xai/data/ImageNet_50K_val"
# ImageNet_dir
# | n01440764/ ...
#  get 2 images for each miniImagenet class and name them as n01440764_1.jpg, n01440764_2.jpg, ...
ImageNet_R_dir = "/home/mario/codes/datasynth-xai/data/Imagenet-R/test"
# ImageNet_R_dir
# | style_1
# | | n01440764/ ...
# get 1 image for each style and each class and name them as style_1_n01440764.jpg, style_2_n01440764.jpg, ...



In [8]:
import os
import shutil
import random
import glob

# Create output directory
output_dir = "/home/mario/codes/promptherder/data/datasets"
os.makedirs(output_dir, exist_ok=True)

# Function to copy and rename files
def copy_and_rename_files(src_path, dst_path, new_name, n_images=1):
    os.makedirs(dst_path, exist_ok=True)
    
    # Get all image files
    extensions = ['*.jpg', '*.jpeg', '*.png',  '*.JPEG']
    files = []
    for ext in extensions:
        files.extend(glob.glob(os.path.join(src_path, ext)))
    
    # Select random images if needed
    if len(files) > n_images:
        files = random.sample(files, n_images)
    
    # Copy and rename
    for i, file in enumerate(files[:n_images]):
        ext = os.path.splitext(file)[1]
        dst_file = os.path.join(dst_path, f"{new_name}_{i+1}{ext}")
        shutil.copy2(file, dst_file)

# Process CUB dataset - 1 image per class
cub_output = os.path.join(output_dir, "CUB")
os.makedirs(cub_output, exist_ok=True)
for class_dir in sorted(os.listdir(CUB_test_dir)):
    if os.path.isdir(os.path.join(CUB_test_dir, class_dir)):
        class_name = class_dir.split('.')[1]  # Extract class name (class_1, class_2, etc.)
        src_path = os.path.join(CUB_test_dir, class_dir)
        copy_and_rename_files(src_path, cub_output, class_name, 1)

# Process HAM10000 dataset - 20 images per class
ham_output = os.path.join(output_dir, "HAM10000")
os.makedirs(ham_output, exist_ok=True)
for class_dir in sorted(os.listdir(HAM1000_dir)):
    if os.path.isdir(os.path.join(HAM1000_dir, class_dir)):
        src_path = os.path.join(HAM1000_dir, class_dir)
        copy_and_rename_files(src_path, ham_output, class_dir, 20)

# Process ImageNet dataset - 2 images per miniImageNet class
imagenet_output = os.path.join(output_dir, "ImageNet")
os.makedirs(imagenet_output, exist_ok=True)
c = 0
for class_name in miniImagenet_classes:
    class_path = os.path.join(ImageNet_dir, class_name)
    if os.path.isdir(class_path):
        copy_and_rename_files(class_path, imagenet_output, class_name, 2)
        c += 1
print(f"Processed {c} classes from ImageNet.")
# Process ImageNet-R dataset - 1 image per style and class
imagenet_r_output = os.path.join(output_dir, "ImageNet_R")
os.makedirs(imagenet_r_output, exist_ok=True)
style_dirs = [d for d in os.listdir(ImageNet_R_dir) if os.path.isdir(os.path.join(ImageNet_R_dir, d))]
for style_dir in style_dirs:
    style_path = os.path.join(ImageNet_R_dir, style_dir)
    for class_name in miniImagenet_classes:
        class_path = os.path.join(style_path, class_name)
        if os.path.isdir(class_path):
            new_name = f"{style_dir}_{class_name}"
            copy_and_rename_files(class_path, imagenet_r_output, new_name, 1)

print(f"Dataset organization completed. Files saved to {output_dir}")

Processed 100 classes from ImageNet.
Dataset organization completed. Files saved to /home/mario/codes/promptherder/data/datasets


### Embeddings try

In [6]:
p = "/home/mario/codes/promptherder/data/datasets/ImageNet/.cache/embeddings_dift_sd.npz"

import numpy as np
data = np.load(p)
print(data['embeddings'].shape)


(200, 1280, 48, 48)


In [13]:
def split_emb(emb, n_parts=4):
    """Split embeddings into n_parts parts."""
    size = emb.shape[2]
    part_size = size // n_parts
    parts = np.zeros((emb.shape[0], emb.shape[1], n_parts, n_parts), dtype=emb.dtype)
    for i in range(n_parts):
        for j in range(n_parts):
            emb_slice = emb[:, :, i*part_size:(i+1)*part_size, j*part_size:(j+1)*part_size]
            parts[:, :, i, j] = np.mean(emb_slice, axis=(2, 3))
    return parts

from pathlib import Path
import os 
DATASETS_DIR = Path("../data") / "datasets"
N_PARTS = 3  # 3x3=9 parts
for dataset in os.listdir(DATASETS_DIR):
    dataset_dir = DATASETS_DIR / dataset
    cache_dir = dataset_dir / ".cache"
    if not cache_dir.exists():
        continue
    for file in os.listdir(cache_dir):
        if "dift_sd.npz" in file:
            p = cache_dir / file
            print(p)
            data = np.load(p)
            print(data['embeddings'].shape)
            embs = split_emb(data['embeddings'], n_parts=N_PARTS)  # 9 parts
            for i in range(N_PARTS):
                for j in range(N_PARTS):
                    splice = embs[:, :, i, j]  # (N, C)
                    new_p = p.parent / f"embeddings_dift_sd_part{i}{j}.npz"
                    np.savez_compressed(new_p, embeddings=splice, paths=data['paths'])
                    print(f"Saved {new_p} with shape {splice.shape}")


../data/datasets/ImageNet/.cache/embeddings_dift_sd.npz
(200, 1280, 48, 48)
Saved ../data/datasets/ImageNet/.cache/embeddings_dift_sd_part00.npz with shape (200, 1280)
Saved ../data/datasets/ImageNet/.cache/embeddings_dift_sd_part01.npz with shape (200, 1280)
Saved ../data/datasets/ImageNet/.cache/embeddings_dift_sd_part02.npz with shape (200, 1280)
Saved ../data/datasets/ImageNet/.cache/embeddings_dift_sd_part10.npz with shape (200, 1280)
Saved ../data/datasets/ImageNet/.cache/embeddings_dift_sd_part11.npz with shape (200, 1280)
Saved ../data/datasets/ImageNet/.cache/embeddings_dift_sd_part12.npz with shape (200, 1280)
Saved ../data/datasets/ImageNet/.cache/embeddings_dift_sd_part20.npz with shape (200, 1280)
Saved ../data/datasets/ImageNet/.cache/embeddings_dift_sd_part21.npz with shape (200, 1280)
Saved ../data/datasets/ImageNet/.cache/embeddings_dift_sd_part22.npz with shape (200, 1280)
../data/datasets/CUB/.cache/embeddings_dift_sd.npz
(200, 1280, 48, 48)
Saved ../data/datasets/CU